In [1]:
# 0. IMPORT LIBRARIES
# ======================
import pandas as pd
import numpy as np
from pathlib import Path

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, average_precision_score, recall_score

import lightgbm as lgb
import xgboost as xgb
from catboost import CatBoostClassifier

In [2]:
# 1. LOAD PREPROCESSED DATA
# ======================
data_path = Path("../data")

df_final = pd.read_parquet(data_path / "preprocessed_train.parquet")

target = "isFraud"


In [3]:
# 2. SPLIT TRAIN / VAL / TEST
# ======================
train_df, test_df = train_test_split(
    df_final,
    test_size=0.1,
    random_state=42,
    stratify=df_final[target]
)

train_df, val_df = train_test_split(
    train_df,
    test_size=0.1,
    random_state=42,
    stratify=train_df[target]
)

X_train = train_df.drop(columns=[target])
y_train = train_df[target]

X_val = val_df.drop(columns=[target])
y_val = val_df[target]

X_test = test_df.drop(columns=[target])
y_test = test_df[target]


In [4]:
# 3. OPTIMIZE MEMORY
# ======================
X_train = X_train.astype("float32")
X_val = X_val.astype("float32")
X_test = X_test.astype("float32")

In [5]:
# 4. EVALUATION FUNCTION
# ======================
def evaluate(model, X, y, name):
    pred = model.predict(X)

    if hasattr(model, "predict_proba"):
        proba = model.predict_proba(X)[:, 1]
    else:
        proba = model.decision_function(X)

    print("\n========================")
    print("MODEL:", name)
    print(classification_report(y, pred))
    print("AUPRC:", round(average_precision_score(y, proba), 4))
    print("Recall:", round(recall_score(y, pred), 4))

    return average_precision_score(y, proba)

In [6]:
# 5. TRAIN MODELS
# ======================

models = {}

# ---- 1. Logistic Regression ----
lr = LogisticRegression(max_iter=500)
lr.fit(X_train, y_train)
models["Logistic Regression"] = lr


# ---- 2. LightGBM ----
lgb_model = lgb.LGBMClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=64,
    random_state=42
)

lgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric="aucpr",
    callbacks=[lgb.early_stopping(50, verbose=True)]
)

models["LightGBM"] = lgb_model


# ---- 3. XGBoost ----
xgb_model = xgb.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8,
    tree_method="hist",
    eval_metric="aucpr"
)

xgb_model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    verbose=True
)

models["XGBoost"] = xgb_model


# ---- 4. CatBoost ----
cat_model = CatBoostClassifier(
    iterations=2000,
    learning_rate=0.05,
    depth=6,
    loss_function="Logloss",
    verbose=200
)

cat_model.fit(
    X_train, y_train,
    eval_set=(X_val, y_val),
    use_best_model=True
)

models["CatBoost"] = cat_model

/Users/user/venv/lib/python3.11/site-packages/sklearn/linear_model/_logistic.py:406: ConvergenceWarning: lbfgs failed to converge after 500 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=500).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


[LightGBM] [Info] Number of positive: 16737, number of negative: 461600
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.178242 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 19960
[LightGBM] [Info] Number of data points in the train set: 478337, number of used features: 187
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.034990 -> initscore=-3.317077
[LightGBM] [Info] Start training from score -3.317077
Training until validation scores don't improve for 50 rounds
Did not meet early stopping. Best iteration is:
[1980]	valid_0's binary_logloss: 0.0490688
[0]	validation_0-aucpr:0.39044
[1]	validation_0-aucpr:0.42725
[2]	validation_0-aucpr:0.44482
[3]	validation_0-aucpr:0.45177
[4]	validation_0-aucpr:0.45857
[5]	validation_0-aucpr:0.46820
[6]	validation_0-aucpr:0.47019
[7]	validation_0-aucpr:0.47195
[8]	validation_0-aucpr:0.47629
[9]	val

In [7]:
# 6. COMPARE MODELS (VALIDATION)
# ======================
results = {}

for name, model in models.items():
    score = evaluate(model, X_val, y_val, name)
    results[name] = score

print("\n================ FINAL RANKING (VAL) ================")
for k, v in sorted(results.items(), key=lambda x: x[1], reverse=True):
    print(f"{k}: AUPRC = {v:.4f}")

best_model_name = max(results, key=results.get)
best_model = models[best_model_name]

print("\nBEST MODEL:", best_model_name)


MODEL: Logistic Regression
              precision    recall  f1-score   support

           0       0.97      1.00      0.98     51289
           1       0.00      0.00      0.00      1860

    accuracy                           0.97     53149
   macro avg       0.48      0.50      0.49     53149
weighted avg       0.93      0.97      0.95     53149

AUPRC: 0.1762
Recall: 0.0


/Users/user/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/user/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
/Users/user/venv/lib/python3.11/site-packages/sklearn/metrics/_classification.py:1833: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])



MODEL: LightGBM
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     51289
           1       0.96      0.62      0.76      1860

    accuracy                           0.99     53149
   macro avg       0.97      0.81      0.87     53149
weighted avg       0.99      0.99      0.98     53149

AUPRC: 0.8448
Recall: 0.6247

MODEL: XGBoost
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     51289
           1       0.96      0.59      0.73      1860

    accuracy                           0.98     53149
   macro avg       0.97      0.79      0.86     53149
weighted avg       0.98      0.98      0.98     53149

AUPRC: 0.8246
Recall: 0.5892

MODEL: CatBoost
              precision    recall  f1-score   support

           0       0.98      1.00      0.99     51289
           1       0.93      0.48      0.64      1860

    accuracy                           0.98     53149
   macro avg       0

In [8]:
# 7. FINAL TEST EVALUATION
# ======================
print("\n================ TEST RESULT ================")
evaluate(best_model, X_test, y_test, best_model_name)


================ TEST RESULT ================

MODEL: LightGBM
              precision    recall  f1-score   support

           0       0.99      1.00      0.99     56988
           1       0.97      0.63      0.76      2066

    accuracy                           0.99     59054
   macro avg       0.98      0.82      0.88     59054
weighted avg       0.99      0.99      0.98     59054

AUPRC: 0.8506
Recall: 0.6321


0.8506033530458108

## **Kết luận mô hình (Test set)**

Trên tập kiểm tra (test set), mô hình LightGBM đạt hiệu quả rất tốt trong bài toán phát hiện gian lận giao dịch.

Cụ thể, mô hình đạt AUPRC = 0.8506, cho thấy khả năng phân biệt giữa giao dịch gian lận và không gian lận là rất cao, đặc biệt trong bối cảnh dữ liệu mất cân bằng mạnh.

Về khả năng phát hiện gian lận, mô hình đạt Recall = 0.63 đối với lớp Fraud, nghĩa là mô hình có thể phát hiện được khoảng 63% các giao dịch gian lận thực tế. Đây là một mức cải thiện tốt so với các mô hình baseline và các phương pháp cân bằng dữ liệu trước đó.

Tuy nhiên, Precision của lớp Fraud là 0.97, cho thấy mô hình rất ít dự đoán sai một giao dịch bình thường thành gian lận (false positive thấp). Điều này giúp giảm thiểu việc gây ảnh hưởng đến trải nghiệm người dùng hợp lệ.

## **Ý nghĩa trong bài toán thực tế**

Trong bài toán fraud detection, việc ưu tiên thường nghiêng về Recall hơn Precision, vì mục tiêu chính là phát hiện càng nhiều giao dịch gian lận càng tốt để giảm thiểu rủi ro tài chính.

Mô hình LightGBM đạt Recall 0.63 được xem là cân bằng tốt giữa:

- Khả năng phát hiện gian lận (Recall khá cao)
- Độ chính xác cảnh báo (Precision rất cao)
- Khả năng phân biệt tổng thể (AUPRC 0.85 – rất tốt với dữ liệu imbalanced)

**LightGBM là mô hình tốt nhất trong các mô hình đã thử nghiệm, vì:**

- Đạt AUPRC cao nhất (0.85 → hiệu suất phân biệt mạnh)
- Recall tốt hơn các mô hình baseline trước đó
- Precision rất cao → ít cảnh báo sai
- Phù hợp với yêu cầu thực tế của hệ thống fraud detection